mixing check

In [1]:
import numpy as np
import matplotlib.pyplot as plt

class DistributedOptimizer:
    def __init__(self, n_nodes, dim, W):
        self.n = n_nodes
        self.d = dim
        self.W = W  # Mixing matrix (doubly stochastic)

    def solve(self, A_list, b_list, x_init, steps, lr=None):
        raise NotImplementedError

class DGD(DistributedOptimizer):
    def solve(self, A_list, b_list, x_init, steps, lr=0.01):
        x = x_init.copy()  # Shape (n, d)
        history = []
        
        for _ in range(steps):
            # Local gradient step: grad = A_i x_i - b_i
            grads = np.einsum('ijd,jd->id', A_list, x) - b_list
            x = x - lr * grads
            # Consensus step
            x = self.W @ x
            # Track global objective value
            f_val = self._compute_global_loss(x, A_list, b_list)
            history.append(f_val)
            
        return np.array(history)

class EXTRA(DistributedOptimizer):
    def solve(self, A_list, b_list, x_init, steps, lr=0.01):
        x = x_init.copy()
        x_prev = x_init.copy()
        history = []
        
        # Precompute gradients at init for correction term
        grads_prev = np.einsum('ijd,jd->id', A_list, x_prev) - b_list

        for _ in range(steps):
            grads = np.einsum('ijd,jd->id', A_list, x) - b_list
            
            # EXTRA Update: x_{k+1} = W x_k - alpha (grad_k - W grad_{k-1})
            # Note: Simplified formulation for strongly convex quadratic
            diff_grads = grads - self.W @ grads_prev
            x_next = self.W @ x - lr * diff_grads
            
            x_prev, grads_prev = x, grads
            x = x_next
            
            f_val = self._compute_global_loss(x, A_list, b_list)
            history.append(f_val)
            
        return np.array(history)

    def _compute_global_loss(self, x, A_list, b_list):
        # Average loss across nodes
        losses = 0.5 * np.einsum('ijd,jd,jd->i', A_list, x, x) - np.einsum('id,id->i', b_list, x)
        return np.mean(losses)

# Helper to generate mixing matrix (Metropolis-Hastings)
def generate_mixing_matrix(n, seed=42):
    np.random.seed(seed)
    # Simple ring graph for demonstration
    W = np.zeros((n, n))
    for i in range(n):
        W[i, i] = 0.5
        W[i, (i-1)%n] = 0.25
        W[i, (i+1)%n] = 0.25
    return W

def run_experiment():
    n_nodes, dim = 10, 5
    W = generate_mixing_matrix(n_nodes)
    steps = 200
    lr = 0.1
    
    # Example: High similarity (A_i identical)
    A_base = np.random.randn(dim, dim)
    A_base = A_base.T @ A_base + np.eye(dim)
    b_base = np.random.randn(dim)
    
    A_list = np.array([A_base] * n_nodes)
    b_list = np.array([b_base] * n_nodes)
    
    x_init = np.random.randn(n_nodes, dim)
    
    dgd = DGD(n_nodes, dim, W)
    extra = EXTRA(n_nodes, dim, W)
    
    loss_dgd = dgd.solve(A_list, b_list, x_init, steps, lr)
    loss_extra = extra.solve(A_list, b_list, x_init, steps, lr)
    
    plt.semilogy(loss_dgd, label='DGD')
    plt.semilogy(loss_extra, label='EXTRA')
    plt.xlabel('Communication Rounds')
    plt.ylabel('Global Objective Value')
    plt.legend()
    plt.title('Convergence Comparison (Identical Data)')
    plt.show()

if __name__ == "__main__":
    run_experiment()

ValueError: operands could not be broadcast together with remapped shapes [original->remapped]: (10,5,5)->(10,5,5) (10,5)->(5,10) 